# Case study: segmenting the Telco customers with k-means

*Session 11, block 1 practice. Author: course team, licence CC-BY-4.0.*

**Task.** Segment the 7,032 Telco customers with k-means and describe the segments. The churn label is **not** used to build the segments; it is used afterwards to describe them.

Steps:
1. Load the data and choose the columns that define "similar".
2. Scale, fit k-means for several k, and look at the elbow and silhouette.
3. Fit the chosen k and build a profile table on the original scale.
4. Check how stable the segments are.
5. Write a short description of each segment.

Theory: [01-clustering-and-k-means.md](../theory/01-clustering-and-k-means.md).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 120)
URL = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
       "master/data/Telco-Customer-Churn.csv")
telco = pd.read_csv(URL)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")
print(telco["TotalCharges"].isna().sum(), "customers with blank TotalCharges (tenure 0) are dropped")
telco = telco.dropna(subset=["TotalCharges"]).reset_index(drop=True)
telco.shape

## 1. Columns for the segmentation

We start with three numeric columns: tenure, monthly charges and total charges. Two binary service columns make the segments more concrete for marketing: whether the customer has fibre-optic internet and whether they have a month-to-month contract. All five are put on the same scale.

Note that total charges are roughly tenure × monthly charges; three columns thus carry partly the same information. This is acceptable for a first segmentation but worth remembering.

In [ ]:
features = pd.DataFrame({
    "tenure": telco["tenure"],
    "monthly": telco["MonthlyCharges"],
    "total": telco["TotalCharges"],
    "fiber": (telco["InternetService"] == "Fiber optic").astype(int),
    "month_to_month": (telco["Contract"] == "Month-to-month").astype(int),
})
Z = StandardScaler().fit_transform(features)
features.describe().round(2)

## 2. Elbow and silhouette

Fit k-means for k = 2 to 8. Silhouette scores use all 7,032 rows; this takes a few seconds.

In [ ]:
rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
    rows.append({"k": k, "inertia": km.inertia_, "silhouette": silhouette_score(Z, km.labels_)})
choice = pd.DataFrame(rows)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
a1.plot(choice["k"], choice["inertia"], marker="o")
a1.set(xlabel="k", ylabel="inertia", title="Elbow method")
a2.plot(choice["k"], choice["silhouette"], marker="o", color="tab:orange")
a2.set(xlabel="k", ylabel="mean silhouette", title="Silhouette")
plt.tight_layout()
choice.round(3)

**Question.** Is there a clear elbow? Which k has the highest silhouette? Would you choose that k for a marketing team, or a different one? Write your answer here.

## 3. Fit and profile

We use k = 4 below. Change `K` and rerun to compare.

In [ ]:
K = 4
km = KMeans(n_clusters=K, n_init=10, random_state=0).fit(Z)
telco["segment"] = km.labels_

profile = telco.groupby("segment").agg(
    customers=("tenure", "size"),
    tenure=("tenure", "mean"),
    monthly=("MonthlyCharges", "mean"),
    total=("TotalCharges", "mean"),
    fiber=("InternetService", lambda s: (s == "Fiber optic").mean()),
    month_to_month=("Contract", lambda s: (s == "Month-to-month").mean()),
    churn_rate=("Churn", lambda s: (s == "Yes").mean()),   # not used for the clustering
)
profile.round(2)

In [ ]:
# a visual profile: each segment's mean of the standardised features
centres = pd.DataFrame(km.cluster_centers_, columns=features.columns)
ax = centres.T.plot.bar(figsize=(9, 3.5), width=0.8)
ax.axhline(0, color="grey", lw=0.8)
ax.set(ylabel="mean (standard deviations)", title="Segment centres on the standardised scale")
plt.tight_layout()

## 4. Stability

Refit on bootstrap samples and compare with the segmentation above using the adjusted Rand index (1 = identical partition).

In [ ]:
rng = np.random.default_rng(0)
aris = []
for seed in range(10):
    idx = rng.choice(len(Z), size=len(Z), replace=True)
    boot = KMeans(n_clusters=K, n_init=10, random_state=seed).fit(Z[idx])
    aris.append(adjusted_rand_score(km.labels_, boot.predict(Z)))
print("ARI min / median:", np.round([min(aris), np.median(aris)], 2))

## 5. Your description

For each segment, write one sentence a marketing manager would understand (who they are, what they pay, how likely they churn) and one possible action.

| Segment | Description | Possible action |
|---|---|---|
| 0 | ... | ... |
| 1 | ... | ... |
| 2 | ... | ... |
| 3 | ... | ... |

## Exercises

1. Run the notebook with only the three numeric columns and **without** scaling. Describe how the segments change, and why.
2. Try K = 3 and K = 5. Which segmentation is easiest to explain? Is it also the most stable?
3. Add the column `PaymentMethod` (one-hot encoded). Does it change the segments? Is a 0/1 column on the same footing as a standardised numeric column?
4. The churn rate differs strongly between segments. Does that make the segmentation a good churn model? Compare with the logistic regression of Session 6.

In [ ]:
# Exercise 1: your code here
